# 01 - Exploratory data analysis

Coverage, data quality and price levels of the analysis base (`marts.int_analysis_prices`: main period 2018-01-01..2025-10-31, valid rows, suspect-low excluded). All numbers come from saved queries in `analysis/queries/`.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
from mandipulse import viz
from mandipulse.viz import COMMODITIES, COMMODITY_COLORS, INK, INK_2, MUTED, load
pd.set_option("display.width", 160)
viz.style()

In [ ]:
# States use categorical slots 4-7 of the reference palette in fixed order (validated separately).
STATE_COLORS = {"Gujarat": "#eda100", "Madhya Pradesh": "#e87ba4",
                "Maharashtra": "#008300", "Uttar Pradesh": "#4a3aa7"}

## Coverage: markets reporting per month

In [ ]:
cov = load("phase3", "eda_01_markets_reporting_by_month")
cov["month"] = pd.to_datetime(cov["month"])
fig, ax = viz.figure("Markets reporting each month, by state",
                     "Distinct markets with >= 1 valid price (any crop) per month")
for state, g in cov.groupby("state"):
    ax.plot(g["month"], g["n_markets_reporting"], color=STATE_COLORS[state], label=state)
viz.label_line_ends(ax, cov, "month", "n_markets_reporting", "state")
ax.set_ylabel("markets reporting")
ax.set_ylim(0)
viz.legend(ax, ncols=4)
viz.save(fig, "eda_markets_reporting_by_month",
         note="Madhya Pradesh dips in Jan-Sep 2024 (known reporting gap); 2025 thins before the Nov-2025 source change.")
cov.pivot(index="month", columns="state", values="n_markets_reporting").tail(12)

In [ ]:
load("phase3", "eda_02_coverage_by_state_crop")

## Data quality: share of rows flagged invalid, by year

In [ ]:
flags = load("phase2", "02_flag_rates_by_year").dropna(subset=["year"])
flags["year"] = flags["year"].astype(int)
fig, ax = viz.figure("Rows flagged invalid, by year",
                     "% of staging rows failing >= 1 validity flag (order, outlier, unit); suspect-low is not counted")
ax.bar(flags["year"], flags["pct_invalid"], color=viz.SEQUENTIAL[4], width=0.6)
for x, y in zip(flags["year"], flags["pct_invalid"], strict=True):
    ax.annotate(f"{y:.2f}%", (x, y), xytext=(0, 3), textcoords="offset points", ha="center",
                fontsize=8.5, color=INK_2)
ax.set_ylabel("% of rows invalid")
ax.set_xticks(flags["year"])
viz.save(fig, "eda_invalid_rate_by_year")
flags[["year", "n_rows", "n_order", "n_outlier", "n_unit_suspect", "n_invalid", "n_suspect_low", "pct_invalid"]]

## Price levels: monthly median modal price

In [ ]:
px = load("phase3", "eda_03_median_price_monthly")
px["month"] = pd.to_datetime(px["month"])
fig, ax = viz.figure("Monthly median wholesale price by crop",
                     "Median of daily modal prices across all markets, Rs per quintal")
for c in COMMODITIES:
    g = px[px["commodity"] == c]
    ax.plot(g["month"], g["median_modal_rs_qtl"], color=COMMODITY_COLORS[c], label=c)
viz.label_line_ends(ax, px, "month", "median_modal_rs_qtl", "commodity")
ax.set_ylabel("Rs / quintal")
ax.set_ylim(0)
viz.legend(ax)
viz.save(fig, "eda_median_price_monthly",
         note="Tomato spikes (e.g. Jul 2023) and onion spikes (e.g. late 2019, 2020) are visible; prices are not inflation-adjusted.")
px.groupby("commodity")["median_modal_rs_qtl"].describe()